In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv('deliveries2.csv')
matches=pd.read_csv('matches.csv')

In [3]:
df.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,2,2,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150460 entries, 0 to 150459
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          150460 non-null  int64 
 1   inning            150460 non-null  int64 
 2   batting_team      150460 non-null  object
 3   bowling_team      150460 non-null  object
 4   over              150460 non-null  int64 
 5   ball              150460 non-null  int64 
 6   batsman           150460 non-null  object
 7   non_striker       150460 non-null  object
 8   bowler            150460 non-null  object
 9   is_super_over     150460 non-null  int64 
 10  wide_runs         150460 non-null  int64 
 11  bye_runs          150460 non-null  int64 
 12  legbye_runs       150460 non-null  int64 
 13  noball_runs       150460 non-null  int64 
 14  penalty_runs      150460 non-null  int64 
 15  batsman_runs      150460 non-null  int64 
 16  extra_runs        150460 non-null  int

# If I want to find the max runs scored by top 5 batsmans, I would do that using groupby function and apply it on batsman column

In [5]:
runs=df.groupby('batsman')

In [6]:
runs

# If I want to find the number of ball played by V kohli, then I would do this as every row in the data is a ball played by any player.

In [7]:
runs.get_group('V Kohli').shape

(3494, 21)

In [8]:
runs['batsman_runs'].sum()

batsman
A Ashish Reddy     280
A Chandila           4
A Chopra            53
A Choudhary         25
A Flintoff          62
                  ... 
YV Takawale        192
Yashpal Singh       47
Younis Khan          3
Yuvraj Singh      2591
Z Khan             117
Name: batsman_runs, Length: 461, dtype: int64

# If I want to find the max runs scored by top 5 batsmans, I would do that using groupby function and apply it on batsman column

In [9]:
runs['batsman_runs'].sum().sort_values(ascending=False).head()

batsman
SK Raina     4548
V Kohli      4423
RG Sharma    4207
G Gambhir    4132
DA Warner    4014
Name: batsman_runs, dtype: int64

# If I want to find the no of 4's hit by batmans, then

In [16]:
mask=df['batsman_runs']==6
new_df=df[mask]

In [17]:
new_df.shape

(6523, 21)

# Total number of 4's hit in IPL till today

In [18]:
new_df.shape[0]

6523

In [19]:
new_df.groupby('batsman')['batsman_runs'].count().sort_values(ascending=False).head()

batsman
CH Gayle     266
SK Raina     174
RG Sharma    173
DA Warner    160
V Kohli      160
Name: batsman_runs, dtype: int64

# If I want to find the runs scored by V kohli against other teams.


In [20]:
new_df_vk=df[df['batsman']=='V Kohli']

In [23]:
new_df_vk.groupby('bowling_team')['batsman_runs'].sum().sort_values(ascending=False).head(3)

bowling_team
Chennai Super Kings    706
Delhi Daredevils       661
Kings XI Punjab        483
Name: batsman_runs, dtype: int64

# Function to find the runs scored by any player against other teams.

In [31]:
def run_scored(batsman_name):
    bt=df[df['batsman']==batsman_name]
    return bt.groupby('bowling_team')['batsman_runs'].sum().sort_values(ascending=False).index[0]

In [32]:
run_scored('V Kohli')

'Chennai Super Kings'

In [33]:
run_scored('DA Warner')

'Kolkata Knight Riders'

In [35]:
run_scored('G Gambhir')

'Kings XI Punjab'

In [36]:
run_scored('MS Dhoni')

'Royal Challengers Bangalore'

# If we want to find who has the highest strike rate in death overs(16-20), then
# Its basis are:
# 1)Strike Rate=(No of runs/No of balls)*100
# 2)Min batsman 200 balls in over 16-20

In [40]:
#First we need to extract the balls of the death overs
mask2=df['over']>15
death_overs=df[mask2]

In [46]:
all_batsman=death_overs.groupby('batsman')['batsman_runs'].count()
sel_batsman=all_batsman>200

In [55]:
batsman_list=all_batsman[sel_batsman].index.tolist()

In [56]:
# Calculate Strike Rate
# To do that we runs and balls

death_overs['batsman']

93         MC Henriques
94         MC Henriques
95         Yuvraj Singh
96             DJ Hooda
97         Yuvraj Singh
              ...      
150455      Sachin Baby
150456      Sachin Baby
150457    Iqbal Abdulla
150458      Sachin Baby
150459    Iqbal Abdulla
Name: batsman, Length: 33737, dtype: object

In [67]:
mask3=death_overs['batsman'].isin(batsman_list)
final=death_overs[mask3]

In [68]:
runs=final.groupby('batsman')['batsman_runs'].sum()

In [69]:
balls=final.groupby('batsman')['batsman_runs'].count()

In [70]:
sr=(runs/balls)*100

In [73]:
sr.sort_values(ascending=False).head(5)

batsman
AB de Villiers    211.052632
DA Warner         189.473684
DA Miller         186.666667
V Kohli           181.868132
RG Sharma         175.668449
Name: batsman_runs, dtype: float64

# If I want to find the Orange cap holders for all the season.

In [80]:
# Merge
matches
# The problem with matches df is there is no column for batsman runs and the problem with 
# the delivery dataframe is there is no columns for seasons.
# This is where we have to merge two dataframes.

,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,2017,Indore,2017-04-08,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,2017,Bangalore,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
631,632,2016,Raipur,2016-05-22,Delhi Daredevils,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Royal Challengers Bangalore,0,6,V Kohli,Shaheed Veer Narayan Singh International Stadium,A Nand Kishore,BNJ Oxenford,NaN
632,633,2016,Bangalore,2016-05-24,Gujarat Lions,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Royal Challengers Bangalore,0,4,AB de Villiers,M Chinnaswamy Stadium,AK Chaudhary,HDPK Dharmasena,NaN
633,634,2016,Delhi,2016-05-25,Sunrisers Hyderabad,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Sunrisers Hyderabad,22,0,MC Henriques,Feroz Shah Kotla,M Erasmus,C Shamshuddin,NaN
634,635,2016,Delhi,2016-05-27,Gujarat Lions,Sunrisers Hyderabad,Sunrisers Hyderabad,field,normal,0,Sunrisers Hyderabad,0,4,DA Warner,Feroz Shah Kotla,M Erasmus,CK Nandan,NaN


In [88]:
new=df.merge(matches,left_on='match_id',right_on='id')

In [89]:
print(df.shape)
print(matches.shape)

(150460, 21)
(636, 18)


In [94]:
type(new.groupby(['season','batsman'])['batsman_runs'].sum())

pandas.core.series.Series

In [102]:
new.groupby(['season','batsman'])['batsman_runs'].sum().sort_values(ascending=False).reset_index().drop_duplicates(subset='season',keep='first').sort_values('season')[['season','batsman']]

,season,batsman
10,2008,SE Marsh
14,2009,ML Hayden
9,2010,SR Tendulkar
11,2011,CH Gayle
2,2012,CH Gayle
3,2013,MEK Hussey
6,2014,RV Uthappa
17,2015,DA Warner
0,2016,V Kohli
7,2017,DA Warner


# Here we are trying to find out the no of total sixes scored by every team in each of over.

In [3]:
mask5=df['batsman_runs']==6
six=df[mask5]

In [6]:
six.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
10,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,2,4,DA Warner,S Dhawan,A Choudhary,0,...,0,0,0,0,6,0,6,NaN,NaN,NaN
47,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,8,4,MC Henriques,S Dhawan,TM Head,0,...,0,0,0,0,6,0,6,NaN,NaN,NaN
75,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,13,2,Yuvraj Singh,MC Henriques,A Choudhary,0,...,0,0,0,0,6,0,6,NaN,NaN,NaN
89,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,15,3,Yuvraj Singh,MC Henriques,S Aravind,0,...,0,0,0,0,6,0,6,NaN,NaN,NaN
91,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,15,5,MC Henriques,Yuvraj Singh,S Aravind,0,...,0,0,0,0,6,0,6,NaN,NaN,NaN


In [7]:
six.shape

(6523, 21)

In [9]:
six.pivot_table(index='over', columns='batting_team', values='batsman_runs', aggfunc='count')

batting_team,Chennai Super Kings,Deccan Chargers,Delhi Daredevils,Gujarat Lions,Kings XI Punjab,Kochi Tuskers Kerala,Kolkata Knight Riders,Mumbai Indians,Pune Warriors,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Sunrisers Hyderabad
over,,,,,,,,,,,,,,
1,5.0,3.0,14.0,4.0,10.0,2.0,10.0,9.0,5.0,12.0,1.0,NaN,20.0,3.0
2,17.0,21.0,19.0,6.0,19.0,4.0,13.0,17.0,6.0,7.0,NaN,2.0,28.0,12.0
3,37.0,11.0,30.0,14.0,27.0,3.0,28.0,22.0,6.0,13.0,2.0,1.0,40.0,16.0
4,34.0,17.0,41.0,9.0,36.0,3.0,35.0,28.0,7.0,21.0,5.0,1.0,43.0,8.0
5,41.0,27.0,29.0,11.0,30.0,3.0,25.0,43.0,7.0,23.0,8.0,5.0,40.0,17.0
6,43.0,22.0,27.0,12.0,39.0,3.0,34.0,50.0,3.0,24.0,6.0,5.0,29.0,10.0
7,22.0,11.0,20.0,10.0,19.0,2.0,26.0,18.0,5.0,15.0,1.0,4.0,18.0,13.0
8,25.0,13.0,26.0,6.0,35.0,2.0,25.0,23.0,10.0,24.0,NaN,3.0,34.0,19.0
9,23.0,17.0,32.0,11.0,36.0,1.0,27.0,35.0,9.0,20.0,3.0,1.0,47.0,15.0


In [10]:
import seaborn as sns

ModuleNotFoundError: No module named 'seaborn'